# Restaurant AI Assistant

**Mid-term project: Building a Basic AI Knowledge Assistant with Memory**

A digital concierge for a restaurant. It answers guest questions from the restaurant's own files (PDFs and Excel / Google Sheets), and it remembers the conversation so follow-up questions work.

```
PDF files ---------> PyPDFLoader + text splitter ----+
                                                      +--> HuggingFace embeddings --> FAISS indexes
Excel / Google Sheet -> every row becomes text -------+
                                                                      |
Guest question --> Query planner --> Retriever --> Prompt + Groq LLM --> Answer
(notebook or        (uses memory)                                      |
 Gradio chat)            ^---------------- Chat memory <---------------+
```

| Requirement in the project brief | Section |
|---|---|
| 1. Data integration: PDFs and Google Sheets converted to text | 3 and 4 |
| 2. Embeddings (HuggingFace) and FAISS vector database, semantic search | 5 |
| 3. RAG pipeline: query, retrieve, generate with an LLM | 7 |
| 4. Chat memory: buffer memory, optional buffer summary memory | 6, 8 and 9 |
| 4. Remembers questions, uses past context, handles follow-ups | 8 and 9 |
| 5. Testing and evaluation: accuracy and relevance | 10 |
| Simple chat in the notebook (no interface) | 11 |
| Demo chat interface (Gradio) | 12 |

**Before running:** add a Colab secret named `GROQ_API_KEY` (key icon in the left sidebar) and enable notebook access. Then run the cells from top to bottom.

## 1. Setup

### Installing Libraries

In [1]:
!pip install -q --no-warn-conflicts langchain-core langchain-community langchain-text-splitters langchain-groq langchain-huggingface faiss-cpu sentence-transformers pypdf openpyxl pandas gradio ddgs pillow numpy


### Imports

In [2]:
import os
import re
import json
import time
import glob
import random
import getpass
import textwrap

import pandas as pd
import requests

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

/tmp/ipykernel_2748/918701344.py:17: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


### Settings

In [3]:
# Restaurant details (the assistant uses them in its answers)
RESTAURANT_NAME = "Bremer"
RESTAURANT_PHONE = "19113"

# Folders for the uploaded files and the saved FAISS indexes
DATA_DIR = "knowledge_files"
INDEX_DIR = "faiss_indexes"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(INDEX_DIR, exist_ok=True)

# Groq models: a large one for answers, a small and fast one for planning, summaries and grading
# (change the names if your Groq key lists other chat models)
ANSWER_MODEL = "openai/gpt-oss-120b"
FAST_MODEL = "openai/gpt-oss-20b"

# Multilingual embedding model: guests can ask in other languages while the files stay in English
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# PDF chunking and retrieval
CHUNK_SIZE = 800
CHUNK_OVERLAP = 150
TOP_K_SHEET = 8          # rows retrieved from a sheet
TOP_K_PDF = 5            # PDF chunks retrieved
MAX_CONTEXT_CHARS = 24000

# Summary memory: how many recent turns are kept word for word (Section 9 uses 2 to show the effect faster)
KEEP_RECENT_TURNS = 4

# Memory used by the Gradio chat in Section 11: "buffer" (full history) or "summary_buffer"
MEMORY_MODE = "buffer"

## 2. Language Models (Groq)

The large model writes the final answers. The small model plans the search, writes memory summaries and grades the evaluation.

### API Key

In [4]:
# Read the Groq key from Colab secrets, otherwise ask for it
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY") or getpass.getpass("Enter your Groq API key: ")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

Enter your Groq API key: ··········


### Creating the Models

In [5]:
# Check that both models exist for this key (model names on Groq change over time)
from groq import Groq

available_models = {m.id for m in Groq(api_key=GROQ_API_KEY).models.list().data}
for model in (ANSWER_MODEL, FAST_MODEL):
    if model not in available_models:
        raise RuntimeError(f"Model '{model}' is not available for this key. Choose a chat model from this list and update the Settings cell:\n{sorted(available_models)}")


def make_llm(model, temperature=0.0, max_tokens=600, retries=3):
    settings = dict(model=model, temperature=temperature, max_retries=retries)
    if model.startswith("openai/gpt-oss"):
        # Reasoning models spend part of the tokens on thinking, so keep the effort low and leave extra room
        settings["max_tokens"] = max_tokens * 3
        try:
            return ChatGroq(reasoning_effort="low", **settings)
        except Exception:
            return ChatGroq(model_kwargs={"reasoning_effort": "low"}, **settings)
    return ChatGroq(max_tokens=max_tokens, **settings)


# Fast model: planning, summaries, grading
fast_llm = make_llm(FAST_MODEL, temperature=0.0, max_tokens=400)

# Answer model: if its rate limit is reached, the fast model answers instead
answer_llm = make_llm(ANSWER_MODEL, temperature=0.2, max_tokens=600, retries=1).with_fallbacks(
    [make_llm(FAST_MODEL, temperature=0.2, max_tokens=600)]
)

### Connection Test

In [6]:
print("Fast model  :", repr(fast_llm.invoke("Reply with one word: ready").content))
print("Answer model:", repr(answer_llm.invoke("Reply with one word: ready").content))

Fast model  : 'ready'
Answer model: 'ready'


## 3. Loading the Data

The knowledge base comes from files you provide:

- **Unstructured data:** PDF files (restaurant info, policies, dish descriptions).
- **Structured data:** an Excel / CSV file, or a Google Sheet link. Every tab becomes its own collection.

### Uploading Files

In [7]:
# Select all your PDF and Excel / CSV files at once
try:
    from google.colab import files
    uploaded = files.upload()
    for name, content in uploaded.items():
        with open(os.path.join(DATA_DIR, os.path.basename(name)), "wb") as handle:
            handle.write(content)
        try:
            os.remove(name)      # Colab also keeps a copy in the working folder
        except OSError:
            pass
    print("Saved:", list(uploaded.keys()))
except ImportError:
    print(f"Not running in Colab: copy your files into the folder '{DATA_DIR}' and run the next cells.")

Saving Bremer_Allergen_and_Dietary_Guide.pdf to Bremer_Allergen_and_Dietary_Guide.pdf
Saving Bremer_Menu_Descriptions.pdf to Bremer_Menu_Descriptions.pdf
Saving Bremer_Restaurant_Data.xlsx to Bremer_Restaurant_Data.xlsx
Saving Bremer_Restaurant_Info_and_Policie.pdf to Bremer_Restaurant_Info_and_Policie.pdf
Saved: ['Bremer_Allergen_and_Dietary_Guide.pdf', 'Bremer_Menu_Descriptions.pdf', 'Bremer_Restaurant_Data.xlsx', 'Bremer_Restaurant_Info_and_Policie.pdf']


### Google Sheet (Optional)

Paste a Google Sheet link (or just its ID) between the quotes to download it as well. The sheet must be shared as "Anyone with the link: Viewer". Leave it empty to use only the uploaded files.

In [8]:
GOOGLE_SHEET_LINK = ""


def download_google_sheet(link):
    match = re.search(r"/d/([a-zA-Z0-9_-]+)", link)
    sheet_id = match.group(1) if match else link.strip()
    url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=xlsx"
    response = requests.get(url, timeout=60)
    if response.status_code != 200 or not response.content.startswith(b"PK"):
        raise RuntimeError("Could not download the sheet. Check that sharing is set to 'Anyone with the link: Viewer'.")
    path = os.path.join(DATA_DIR, "google_sheet.xlsx")
    with open(path, "wb") as handle:
        handle.write(response.content)
    return path


if GOOGLE_SHEET_LINK.strip():
    print("Google Sheet downloaded to:", download_google_sheet(GOOGLE_SHEET_LINK))
else:
    print("No Google Sheet link provided: only the uploaded files will be used.")

No Google Sheet link provided: only the uploaded files will be used.


### Files Inventory

In [9]:
all_files = sorted(glob.glob(os.path.join(DATA_DIR, "*")))
pdf_paths = [p for p in all_files if p.lower().endswith(".pdf")]
sheet_paths = [p for p in all_files if p.lower().endswith((".xlsx", ".xlsm", ".csv"))]

print("PDF files   :", [os.path.basename(p) for p in pdf_paths])
print("Sheet files :", [os.path.basename(p) for p in sheet_paths])

if not pdf_paths or not sheet_paths:
    raise RuntimeError("The project needs both data sources: at least one PDF and one Excel / CSV file (or a Google Sheet). Upload them in Section 3 and run this cell again.")

PDF files   : ['Bremer_Allergen_and_Dietary_Guide.pdf', 'Bremer_Menu_Descriptions.pdf', 'Bremer_Restaurant_Info_and_Policie.pdf']
Sheet files : ['Bremer_Restaurant_Data.xlsx']


### Reading the Sheets

In [10]:
# Read every tab of every sheet file into a pandas table
sheet_tables = {}     # (file name, tab name) -> table

for path in sheet_paths:
    file_name = os.path.basename(path)
    if path.lower().endswith(".csv"):
        frames = {os.path.splitext(file_name)[0]: pd.read_csv(path)}
    else:
        workbook = pd.ExcelFile(path)
        frames = {tab: workbook.parse(tab) for tab in workbook.sheet_names}

    for tab, frame in frames.items():
        frame = frame.dropna(how="all").dropna(axis=1, how="all")
        frame.columns = [str(c).strip() for c in frame.columns]
        if not frame.empty:
            sheet_tables[(file_name, tab)] = frame

for (file_name, tab), frame in sheet_tables.items():
    print(f"{file_name} / {tab}: {frame.shape[0]} rows, {frame.shape[1]} columns -> {list(frame.columns)}")

Bremer_Restaurant_Data.xlsx / Menu: 111 rows, 15 columns -> ['item_id', 'name_en', 'name_ar', 'category', 'price_egp', 'currency', 'price_source', 'description', 'calories_kcal', 'is_vegetarian', 'spice_level', 'allergen_hints', 'is_available', 'notes', 'image_url']
Bremer_Restaurant_Data.xlsx / Offers: 4 rows, 9 columns -> ['offer_id', 'offer_name', 'description', 'discount_percent', 'valid_days', 'valid_time', 'branch', 'conditions', 'status']
Bremer_Restaurant_Data.xlsx / Branches: 2 rows, 9 columns -> ['branch_id', 'branch_name', 'address', 'city', 'opening_days', 'opening_hours', 'ordering_phone', 'services', 'status']


## 4. Data Integration

Everything is converted into LangChain `Document` objects (text plus metadata).

- **Sheet rows** become short sentences of the form `column: value`, so tables can be searched by meaning. Each row keeps its file, tab and row number in its metadata.
- **PDF pages** are split into overlapping chunks. Each chunk keeps its file name and page number in its metadata.

### Row to Text Functions

In [11]:
def clean_value(value):
    # Turn a cell into clean text (None for empty cells)
    if value is None or pd.isna(value):
        return None
    if isinstance(value, pd.Timestamp):
        return value.strftime("%Y-%m-%d")
    if isinstance(value, float):
        return str(int(value)) if value.is_integer() else f"{value:g}"
    text = str(value).strip().rstrip(" .")
    return text or None


def pick_title_column(frame):
    # The column that names each record: the first one containing "name" or "title", otherwise the first column
    for column in frame.columns:
        if "name" in column.lower() or "title" in column.lower():
            return column
    return frame.columns[0]


SKIP_COLUMNS = {"image_url"}      # stored in metadata only, never embedded or sent to the LLM


def row_to_text(file_name, tab, row):
    parts = []
    for column, value in row.items():
        if column in SKIP_COLUMNS:
            continue
        text = clean_value(value)
        if text is not None:
            parts.append(f"{column}: {text}")
    return f"Record from sheet '{tab}' ({file_name}). " + ". ".join(parts) + "."


def compact_table(frame, max_text=50):
    # The whole table in a few lines: header once, then one line per row (long text columns are left out).
    # It is used for questions such as "cheapest" or "list all", where searching a few rows is not enough.
    keep = [c for c in frame.columns if c not in SKIP_COLUMNS and frame[c].astype(str).str.len().mean() <= max_text]
    lines = [" | ".join(keep)]
    for _, row in frame[keep].iterrows():
        lines.append(" | ".join(clean_value(row[c]) or "-" for c in keep))
    return "\n".join(lines)

### Sheet Collections

In [12]:
# One collection per sheet tab: its documents, plus a short description used by the query planner
COLLECTIONS = {}

for (file_name, tab), frame in sheet_tables.items():
    name = re.sub(r"[^a-z0-9]+", "_", tab.lower()).strip("_") or "sheet"
    if name in COLLECTIONS:
        name = re.sub(r"[^a-z0-9]+", "_", file_name.lower()).strip("_") + "_" + name

    title_column = pick_title_column(frame)
    docs = []
    for index, row in frame.iterrows():
        sheet_row = int(index) + 2      # row 1 of the sheet holds the column names
        docs.append(Document(
            page_content=row_to_text(file_name, tab, row),
            metadata={
                "source_type": "sheet",
                "title": clean_value(row[title_column]) or f"row {sheet_row}",
                "label": f"{file_name}, tab '{tab}', row {sheet_row}",
                "image_url": clean_value(row["image_url"]) if "image_url" in frame.columns else None,
                "category": clean_value(row["category"]) if "category" in frame.columns else None,
            },
        ))

    # rows grouped by category: lets the retriever load a whole category (all drinks, all sandwiches...)
    categories = {}
    for doc in docs:
        if doc.metadata.get("category"):
            categories.setdefault(doc.metadata["category"], []).append(doc)

    full_table = Document(
        page_content=f"Complete table of sheet '{tab}' ({file_name}), {len(docs)} rows, long text columns omitted. Columns separated by |.\n" + compact_table(frame),
        metadata={"source_type": "sheet", "title": tab, "label": f"{file_name}, tab '{tab}' (full table)"},
    )

    COLLECTIONS[name] = {
        "kind": "sheet", "docs": docs, "categories": categories, "full_table": full_table, "source": f"{file_name} / {tab}",
        "description": f"Spreadsheet '{file_name}', tab '{tab}', {len(docs)} rows. Columns: {', '.join(frame.columns)}. Example record: {docs[0].page_content[:220]}",
    }

# Image lookup used by the chat interface, keyed by the item_id of the exact menu row.
# Matching by id (never by dish name) keeps same-named dishes and look-alike names apart.
IMAGE_OVERRIDES = {}      # item_id -> corrected link, if the photo check finds a wrong photo
MENU_ITEMS = {}
for (file_name, tab), frame in sheet_tables.items():
    if not {"item_id", "image_url"} <= set(frame.columns):
        continue
    for _, row in frame.iterrows():
        item_id, url = clean_value(row["item_id"]), clean_value(row["image_url"])
        if item_id:
            item_id = item_id.upper()
            MENU_ITEMS[item_id] = {
                "id": item_id,
                "name": clean_value(row.get("name_en")) or item_id,
                "name_ar": clean_value(row.get("name_ar")) or "",
                "category": clean_value(row.get("category")) or "",
                "price": clean_value(row.get("price_egp")) or "",
                "url": IMAGE_OVERRIDES.get(item_id, url) or "",
            }
ALL_CATEGORIES = sorted({c for info in COLLECTIONS.values() for c in info.get("categories", {})})
print(f"Menu items: {len(MENU_ITEMS)} | Menu categories: {ALL_CATEGORIES}")

print("Example record:")
print(next(iter(COLLECTIONS.values()))["docs"][0].page_content)

Menu items: 111 | Menu categories: ['Appetizers', 'Asian', 'Beef', 'Chicken', 'Combo', 'Desserts', 'Oriental Meals', 'Pasta', 'Picks for you', 'Pizza', 'Rice Dishes', 'Salads', 'Sandwiches', 'Sauces', 'Side items', 'Soft Drinks']
Example record:
Record from sheet 'Menu' (Bremer_Restaurant_Data.xlsx). item_id: BR001. name_en: French Fries. name_ar: فرنش فرايز. category: Picks for you. price_egp: 45. currency: EGP. price_source: EGMenus. description: French Fries. calories_kcal: 365. is_vegetarian: Yes. spice_level: 0. allergen_hints: none identified from the listed ingredients. is_available: Yes. notes: Calories are estimates, not verified nutrition data. Vegetarian and allergen fields are derived from the listed ingredients, not an official declaration. Availability is a demo value (no live stock feed).


### PDF Loading and Chunking

In [13]:
# Menu items that no longer exist: removed from the PDF text too (the Excel rows are already deleted)
REMOVED_ITEMS = ["Grilled Chicken Breasts Rice Dish", "Chicken Breasts Meal"]


def remove_items(text, names=REMOVED_ITEMS):
    for name in names:
        flex = r"\s+".join(map(re.escape, name.split()))      # the name may wrap onto two lines
        text = re.sub(rf"(?m)^[ \t]*{flex}\s*:.*?Availability:[^.]*\.[ \t]*\n?", "", text, flags=re.S)      # its menu entry
        text = re.sub(rf"(?<!\w){flex}\s*,\s*", "", text)                                                  # inside a list
        text = re.sub(rf",\s*{flex}(?=\s*[.,])", "", text)                                                  # last in a list
    return text


# Load the PDF pages and keep only PDFs that contain text
pdf_pages = []
pdf_intro = []
for path in pdf_paths:
    pages = PyPDFLoader(path).load()
    for page in pages:
        page.page_content = remove_items(page.page_content)
    text = " ".join(p.page_content for p in pages).strip()
    if not text:
        print("WARNING: no text found (scanned PDF?), skipped:", os.path.basename(path))
        continue
    pdf_pages.extend(pages)
    pdf_intro.append(f"{os.path.basename(path)} (starts: {' '.join(text.split())[:140]})")

if not pdf_pages:
    raise RuntimeError("None of the PDF files contains readable text. Scanned PDFs need OCR first.")

# Split the pages into overlapping chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)
pdf_chunks = splitter.split_documents(pdf_pages)

# Add file name and page number to the metadata
for chunk in pdf_chunks:
    file_name = os.path.basename(chunk.metadata.get("source", "pdf"))
    page_number = int(chunk.metadata.get("page", 0)) + 1
    chunk.metadata["source_type"] = "pdf"
    chunk.metadata["label"] = f"{file_name}, page {page_number}"

COLLECTIONS["documents"] = {
    "kind": "pdf", "docs": pdf_chunks, "source": "PDF files",
    "description": "PDF documents: " + " | ".join(pdf_intro),
}

print(f"PDF pages: {len(pdf_pages)} | Chunks: {len(pdf_chunks)}\n")
print(pdf_chunks[0].metadata["label"])
print(pdf_chunks[0].page_content[:300])

PDF pages: 16 | Chunks: 68

Bremer_Allergen_and_Dietary_Guide.pdf, page 1
Bremer Restaurant: Allergen and Dietary Guide
Important safety statement
Bremer has not published a complete official allergen table. The information in this guide was derived
from the ingredients listed for each dish. It is not an official allergen declaration and it can be
incomplete.
A dish that 


### Collections Overview

In [14]:
pd.DataFrame(
    [{"collection": name, "source": info["source"], "type": info["kind"], "documents": len(info["docs"])} for name, info in COLLECTIONS.items()]
)

,collection,source,type,documents
0,menu,Bremer_Restaurant_Data.xlsx / Menu,sheet,111
1,offers,Bremer_Restaurant_Data.xlsx / Offers,sheet,4
2,branches,Bremer_Restaurant_Data.xlsx / Branches,sheet,2
3,documents,PDF files,pdf,68


## 5. Embedding and Indexing

Each document becomes a vector with a HuggingFace sentence-transformer model. The vectors are stored in FAISS, one index per collection, and saved to disk.

### Embedding Model

In [15]:
# Runs locally, no API key needed
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    encode_kwargs={"normalize_embeddings": True},
)
print("Embedding dimension:", len(embeddings.embed_query("test sentence")))

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 9.08MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimension: 384


### Vector Database (FAISS)

In [16]:
VECTOR_STORES = {}

for name, info in COLLECTIONS.items():
    store = FAISS.from_documents(info["docs"], embeddings)
    store.save_local(os.path.join(INDEX_DIR, name))
    VECTOR_STORES[name] = store
    print(f"Index built: {name:<15} vectors: {store.index.ntotal}")

Index built: menu            vectors: 111
Index built: offers          vectors: 4
Index built: branches        vectors: 2
Index built: documents       vectors: 68


### Semantic Search Test

In [17]:
# Reload one index from disk to prove that it was saved correctly
first_name = next(iter(COLLECTIONS))
reloaded = FAISS.load_local(os.path.join(INDEX_DIR, first_name), embeddings, allow_dangerous_deserialization=True)
print(f"Reloaded index '{first_name}' from disk, vectors: {reloaded.index.ntotal}\n")

# Search every collection (a smaller distance means a closer meaning)
for query in ["opening hours", "price", "vegetarian dishes"]:
    print("Query:", query)
    for name, store in VECTOR_STORES.items():
        doc, distance = store.similarity_search_with_score(query, k=1)[0]
        print(f"  [{name}] distance {distance:.3f} | {doc.metadata['label']}")
    print()

Reloaded index 'menu' from disk, vectors: 111

Query: opening hours
  [menu] distance 1.928 | Bremer_Restaurant_Data.xlsx, tab 'Menu', row 49
  [offers] distance 1.340 | Bremer_Restaurant_Data.xlsx, tab 'Offers', row 4
  [branches] distance 1.153 | Bremer_Restaurant_Data.xlsx, tab 'Branches', row 3
  [documents] distance 0.855 | Bremer_Restaurant_Info_and_Policie.pdf, page 1

Query: price
  [menu] distance 1.044 | Bremer_Restaurant_Data.xlsx, tab 'Menu', row 71
  [offers] distance 1.423 | Bremer_Restaurant_Data.xlsx, tab 'Offers', row 2
  [branches] distance 1.800 | Bremer_Restaurant_Data.xlsx, tab 'Branches', row 2
  [documents] distance 0.867 | Bremer_Restaurant_Info_and_Policie.pdf, page 2

Query: vegetarian dishes
  [menu] distance 0.918 | Bremer_Restaurant_Data.xlsx, tab 'Menu', row 49
  [offers] distance 1.512 | Bremer_Restaurant_Data.xlsx, tab 'Offers', row 3
  [branches] distance 1.990 | Bremer_Restaurant_Data.xlsx, tab 'Branches', row 2
  [documents] distance 0.539 | Bremer_Al

## 6. Conversation Memory

Memory is a small dictionary with the chat history (a list of turns) and a summary text.

- **Buffer memory:** keeps the full conversation history (main requirement).
- **Summary buffer memory (optional):** keeps the last few turns word for word and compresses older turns into a summary written by the LLM, so long conversations stay short.

The query planner uses the memory to rewrite follow-up questions, and the answer prompt receives it so replies stay consistent.

### Buffer Memory

In [18]:
def new_memory(keep_recent=KEEP_RECENT_TURNS):
    # history: list of {"user": ..., "assistant": ...} | summary: text of the older turns (summary mode only)
    # keep_recent: number of recent turns kept word for word in summary mode
    return {"history": [], "summary": "", "keep_recent": keep_recent}


def save_turn(memory, question, answer, mode="buffer"):
    memory["history"].append({"user": question, "assistant": answer})
    if mode == "summary_buffer":
        compress_memory(memory)


def history_to_messages(history):
    # Convert saved turns into chat messages for the prompt
    messages = []
    for turn in history:
        messages.append(HumanMessage(content=turn["user"]))
        messages.append(AIMessage(content=turn["assistant"]))
    return messages

### Summary Buffer Memory

In [19]:
SUMMARY_PROMPT = ChatPromptTemplate.from_template(
    """You maintain a running summary of a conversation between a guest and a restaurant assistant.
Update the summary with the new lines. Keep what matters for later: the guest's occasion, party size, budget,
dietary needs and allergies, dishes and wines discussed, prices mentioned and decisions made.
Write at most 120 words in plain English.

Current summary: {summary}

New lines:
{transcript}

Updated summary:"""
)
summary_chain = SUMMARY_PROMPT | fast_llm | StrOutputParser()


def compress_memory(memory):
    # Move the older turns into the summary and keep only the recent turns word for word
    keep_recent = memory["keep_recent"]
    if len(memory["history"]) <= keep_recent:
        return
    old_turns = memory["history"][:-keep_recent]
    recent_turns = memory["history"][-keep_recent:]

    transcript = "\n".join(f"Guest: {t['user']}\nAssistant: {t['assistant']}" for t in old_turns)
    try:
        memory["summary"] = summary_chain.invoke({"summary": memory["summary"] or "(empty)", "transcript": transcript}).strip()
        memory["history"] = recent_turns
    except Exception:
        pass      # if summarising fails, keep the full history for now

## 7. Retrieval-Augmented Generation Pipeline

For every guest message:

1. **Plan:** the fast model rewrites the question so it stands alone (using memory), chooses the collections to search, and says whether the question needs a whole list (cheapest, list all).
2. **Retrieve:** search the chosen FAISS indexes. Whole-list questions read the full sheet table instead.
3. **Generate:** the answer model answers only from the retrieved context, memory and question.
4. **Finalize:** an allergy note is added when needed, and the turn is saved in memory.

### Prompts

In [20]:
PLANNER_SYSTEM = """You are the query planner of a restaurant assistant. Read the conversation and the guest's latest message, then return a JSON object with exactly these keys:

"standalone_question": the latest message rewritten in English as a complete question that can be understood without the conversation. Resolve references such as "it", "that one", "the cheaper one" or "them" using the conversation.
"collections": a list with one or more collection names, copied exactly from the catalog below.
"categories": a list of menu category names, copied exactly from the list of menu categories below, that the guest asks about or whose items are needed to answer: for example the drinks category when the guest wants something to drink, the sandwiches category when a sandwich is requested, the pizza category for a pizza. A request with several parts (a pizza, a rice dish, a sandwich and a drink) needs one category per part. Use an empty list when the question is not about menu items.
"aggregate": true if answering requires scanning a whole list (cheapest, most expensive, how many, list all, compare all), otherwise false.

Catalog of collections:
{catalog}

Menu categories: {categories}

Choose every collection that is needed. Questions about policies, opening hours, reservations, allergens, dietary needs or dish details usually need the PDF collection. Questions about prices, items, wines or offers usually need the matching sheet collection. When unsure, choose several.
Return only the JSON object and nothing else."""

PLANNER_PROMPT = ChatPromptTemplate.from_messages([
    ("system", PLANNER_SYSTEM),
    ("human", "Conversation so far:\n{history}\n\nLatest guest message: {question}"),
])

ANSWER_SYSTEM = """You are the virtual concierge of {restaurant}, a restaurant.

Rules:
1. Answer using only the CONTEXT below. If the context does not contain the answer, say that you do not have that information and suggest {contact_hint}. Never invent dishes, prices, hours or policies.
2. Reply in the same language as the guest's latest message.
3. Use the conversation summary and the chat history to understand references such as "it", "that one" or "the cheaper option".
4. Be warm, polished and concise: one to four sentences, or a short list when several items are needed. Give prices with the currency used in the context.
5. If an item is marked as unavailable or sold out, say so and suggest a similar available alternative from the context.
6. For questions about the cheapest, most expensive, highest, lowest, how many or "list all", read every row of the table in the context carefully before answering.
7. Never say that a dish is safe for an allergy. Allergen and calorie information may be estimated: say so when it is relevant.
8. If the guest greets you, greet them back and offer your help. If the question is unrelated to {restaurant}, politely explain that you can only help with questions about the restaurant.

9. Every time your reply names a menu item that the guest can order (dish, drink, side, sauce), including when you only give its price or compare it, write the item_id of the exact context row in square brackets directly after the name, for example: Grilled Kofta Rice Dish [BR028]. Tag each item once, never invent an ID, and use the ID of the exact row you mean, because the same dish name can exist in several categories with different prices. Do NOT tag or list the ingredients inside a dish (the rice, sauce or kofta of a rice dish) as separate items.
10. If the guest asks for a combination (for example rice and kofta), first look for ONE menu item that is exactly that combination (for example in the Rice Dishes category) and recommend it. Describe what it comes with in plain words, and offer separate items only if the guest asks for more options or no exact match exists.
11. Never say you have no information about a kind of item (drinks, sandwiches, salads, desserts...) when the CONTEXT contains items of that kind: use them. When the guest asks you to choose for them, pick real items from the CONTEXT for every part of the request (for example one pizza, one rice dish, one sandwich and one drink) and give the total price.
12. Never use emojis, emoticons or decorative symbols. Plain text and simple markdown only.

Summary of the earlier conversation (may be empty): {summary}

CONTEXT:
{context}"""

ANSWER_PROMPT = ChatPromptTemplate.from_messages([
    ("system", ANSWER_SYSTEM),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}"),
])

planner_chain = PLANNER_PROMPT | fast_llm | StrOutputParser()
answer_chain = ANSWER_PROMPT | answer_llm | StrOutputParser()

### Step 1: Query Planner

In [21]:
CATALOG = "\n".join(f"- {name}: {info['description']}" for name, info in COLLECTIONS.items())


def history_for_planner(memory, max_turns=3, max_chars=300):
    # Short text version of the memory for the planner
    lines = []
    if memory["summary"]:
        lines.append(f"Summary of earlier conversation: {memory['summary']}")
    for turn in memory["history"][-max_turns:]:
        lines.append(f"Guest: {turn['user'][:max_chars]}")
        lines.append(f"Assistant: {turn['assistant'][:max_chars]}")
    return "\n".join(lines) if lines else "(no previous messages)"


def make_plan(question, memory):
    # If the planner fails, search every collection with the original question
    plan = {"standalone_question": question, "collections": list(COLLECTIONS), "aggregate": False, "categories": []}
    try:
        raw = planner_chain.invoke({
            "history": history_for_planner(memory), "question": question, "catalog": CATALOG, "categories": ", ".join(ALL_CATEGORIES),
        })
        data = json.loads(re.search(r"\{.*\}", raw, flags=re.DOTALL).group(0))
    except Exception:
        return plan

    chosen = data.get("collections", [])
    if isinstance(chosen, str):
        chosen = [chosen]
    chosen = [c for c in chosen if c in COLLECTIONS]

    wanted = data.get("categories", [])
    if isinstance(wanted, str):
        wanted = [wanted]
    by_lower = {c.lower(): c for c in ALL_CATEGORIES}
    plan["categories"] = [by_lower[str(c).strip().lower()] for c in wanted if str(c).strip().lower() in by_lower]

    plan["standalone_question"] = str(data.get("standalone_question") or question)
    plan["collections"] = chosen or list(COLLECTIONS)
    plan["aggregate"] = str(data.get("aggregate", False)).strip().lower() == "true"
    return plan

### Step 2: Retriever

In [22]:
CATEGORY_ROWS = 15      # most rows loaded for one requested category


def retrieve(plan):
    # Search the collections chosen by the planner and merge the results without duplicates.
    # Rows of the requested menu categories come first, so a request such as "and a drink" always sees every drink.
    documents, seen = [], set()

    def add(doc):
        key = (doc.metadata["label"], doc.page_content)
        if key not in seen:
            seen.add(key)
            documents.append(doc)

    for name in plan["collections"]:
        info = COLLECTIONS[name]
        for category in plan.get("categories", []):
            for doc in info.get("categories", {}).get(category, [])[:CATEGORY_ROWS]:
                add(doc)

    for name in plan["collections"]:
        info = COLLECTIONS[name]
        store = VECTOR_STORES[name]

        if plan["aggregate"] and info["kind"] == "sheet":
            if len(info["full_table"].page_content) <= MAX_CONTEXT_CHARS:
                candidates = [info["full_table"]]      # the whole sheet as one compact table
            else:
                candidates = store.similarity_search(plan["standalone_question"], k=min(40, store.index.ntotal))
        else:
            k = TOP_K_PDF if info["kind"] == "pdf" else TOP_K_SHEET
            candidates = store.similarity_search(plan["standalone_question"], k=min(k, store.index.ntotal))

        for doc in candidates:
            add(doc)
    return documents


def build_context(documents):
    # Join the retrieved documents into one context text, up to the size limit
    blocks, used = [], 0
    for doc in documents:
        if used + len(doc.page_content) > MAX_CONTEXT_CHARS:
            break
        blocks.append(doc.page_content)
        used += len(doc.page_content)
    return "\n\n".join(blocks)

### Allergy Note and Contact Hint

Settings used in step 4 (finalize) and in the answer prompt.

In [23]:
ALLERGY_WORDS = re.compile(
    r"\b(allerg\w*|intoleran\w*|coeliac|celiac|anaphyla\w*|gluten|nuts?|peanuts?|shellfish|lactose|dairy|sesame|soy|sulphites?)\b",
    re.IGNORECASE,
)
ALLERGY_NOTE = (
    "Allergy note: ingredient information is not an official allergen declaration. "
    "Please confirm any allergy or intolerance with the restaurant before ordering."
)
CONTACT_HINT = f"calling the restaurant on {RESTAURANT_PHONE}" if RESTAURANT_PHONE else "contacting the restaurant directly"

### Steps 3 and 4: Generate, Finalize and Remember (The Assistant Function)

In [24]:
import unicodedata

ITEM_TAG = re.compile(r"\s*[\[(]\s*(?:[A-Za-z]{1,4}\d{2,5}\s*[,;/]?\s*)+[\])]")
ITEM_TOKEN = re.compile(r"[A-Za-z]{1,4}\d{2,5}")


SEPARATOR = r"(?:[\W_]|\band\b)+"
SMALL_WORDS = {"a", "la", "de", "and", "the", "of", "with"}
ITEM_VARIANTS = {}      # item_id -> the word lists a dish name may be written as


def fold(text):
    # lower case, accents removed, same length as the original text
    return "".join(unicodedata.normalize("NFKD", ch)[0] for ch in str(text)).lower()


def item_variants(item_id):
    if item_id not in ITEM_VARIANTS:
        item, variants = MENU_ITEMS[item_id], set()
        for name in (item["name"], item["name_ar"]):
            for form in (name, re.sub(r"\(.*?\)", " ", name or "")):      # with and without "(Spicy)"
                words = tuple(w for w in re.findall(r"\w+", fold(form)) if w != "and")
                if len(" ".join(words)) > 3:
                    variants.add(words)
        ITEM_VARIANTS[item_id] = variants
    return ITEM_VARIANTS[item_id]


def find_mentions(clean):
    # Every place where a menu item name is written in the text: list of (start, end, [item ids]).
    # Longest names first, so "Grilled Kofta Rice Dish" is one mention and not also "Grilled Kofta".
    plain = fold(clean)
    if len(plain) != len(clean):
        plain = clean.lower()
    by_words = {}
    for item_id in MENU_ITEMS:
        for words in item_variants(item_id):
            by_words.setdefault(words, []).append(item_id)
    taken, mentions = [], []
    for words in sorted(by_words, key=lambda w: -len(" ".join(w))):
        # Arabic text often glues a letter to the next word ("وGrilled", "والمياه"): only Latin neighbours block a Latin name
        latin = words[0][0].isascii()
        pattern = (r"(?<![a-z0-9])" if latin else r"(?<![\u0600-\u06FF])") + SEPARATOR.join(map(re.escape, words)) + (r"(?![a-z0-9])" if words[-1][-1].isascii() else r"(?![\u0600-\u06FF])")
        for match in re.finditer(pattern, plain):
            if any(match.start() < end and start < match.end() for start, end in taken):
                continue
            taken.append((match.start(), match.end()))
            mentions.append((match.start(), match.end(), by_words[words]))
    return sorted(mentions)


def is_presented(clean, start, end):
    """True when the name is offered to the guest as an item (bold, a list entry, followed by its price,
    or written exactly like a menu name). False for words inside a description such as
    "yellow rice topped with charcoal-grilled kofta"."""
    before, after = clean[:start], clean[end:end + 24]
    if re.search(r"\*\*\s*$", before):
        return True
    if re.search(r"(?:^|\n)[ \t]*(?:[-*•]|\d+[.)])[ \t]*\**[ \t]*$", before):
        return True
    if re.match(r"^[\s*)\]]*(?:[-–—:][\s*]*)?\d", after):
        return True
    tokens = re.findall(r"\w+", clean[start:end])
    return any(t[0].isupper() for t in tokens) and all(t[0].isupper() or t.lower() in SMALL_WORDS for t in tokens)


def dishes_in_answer(answer, context):
    """Return (answer without id tags, item_ids of every menu item the answer offers, in reading order).

    1. ids the model wrote itself (only ids that are really in the retrieved context count);
    2. names written as items in the text (not ingredients inside a description).
    A short or ambiguous name counts only if that row was retrieved for this question.
    """
    context_ids = [i for i in MENU_ITEMS if re.search(rf"(?<![A-Za-z0-9]){i}(?![A-Za-z0-9])", context)]
    clean = ITEM_TAG.sub("", answer).strip()
    mentions = find_mentions(clean)

    chosen = []
    for tag in ITEM_TAG.findall(answer):
        for token in ITEM_TOKEN.findall(tag):
            token = token.upper()
            if token in context_ids and token not in chosen:
                chosen.append(token)

    for start, end, ids in mentions:
        if not is_presented(clean, start, end):
            continue
        in_context = [i for i in ids if i in context_ids]
        if in_context:
            pick = in_context[:1] if len({MENU_ITEMS[i]["url"] for i in in_context}) == 1 else in_context
        elif len(fold(clean[start:end]).split()) > 1 and len({MENU_ITEMS[i]["url"] for i in ids}) == 1:
            pick = ids[:1]
        else:
            pick = []      # a single-word or ambiguous name that was not retrieved: do not guess
        chosen.extend(i for i in pick if i not in chosen)

    first_seen = {}
    for start, _, ids in mentions:
        for i in ids:
            first_seen.setdefault(i, start)
    chosen.sort(key=lambda i: first_seen.get(i, 10**9))

    shown, urls = [], set()
    for i in chosen:
        url = MENU_ITEMS[i]["url"]
        if url and url not in urls:
            urls.add(url)
            shown.append(i)
    return clean, shown


EMOJIS = re.compile("[\U0001F000-\U0001FAFF\U0001F1E6-\U0001F1FF\u2300-\u23FF\u2600-\u27BF\u2B00-\u2BFF\uFE0F\u200D]+[ \t]?")


def strip_emojis(text):
    return EMOJIS.sub("", text).strip()


def ask(question, memory, mode="buffer"):
    # One full turn: plan, retrieve, generate, remember
    started = time.time()

    plan = make_plan(question, memory)
    documents = retrieve(plan)
    context = build_context(documents)

    raw_answer = answer_chain.invoke({
        "restaurant": RESTAURANT_NAME,
        "contact_hint": CONTACT_HINT,
        "summary": memory["summary"] or "(none)",
        "context": context or "(no relevant context found)",
        "history": history_to_messages(memory["history"]),
        "question": question,
    }).strip()
    raw_answer = strip_emojis(raw_answer)      # no emojis in the reply or in the memory

    # The memory keeps the ids (so follow-ups such as "the second one" point to the exact row);
    # the guest sees the answer without them
    answer, item_ids = dishes_in_answer(raw_answer, context)
    save_turn(memory, question, raw_answer, mode)
    if ALLERGY_WORDS.search(question) or ALLERGY_WORDS.search(plan["standalone_question"]):
        answer = answer + "\n\n" + ALLERGY_NOTE

    return {"answer": answer, "item_ids": item_ids, "seconds": time.time() - started}


def chat(question, memory, mode="buffer"):
    # Print one turn of the conversation
    reply = ask(question, memory, mode)
    print(f"Guest: {question}")
    print("Assistant:")
    print(textwrap.indent(reply["answer"], "    "))
    print()
    return reply


## 8. Demo: Buffer Memory (Full History)

The guest never repeats the dish name. Questions such as "How much is it?" only work because the assistant remembers the previous turns.

The first question uses a real item from your uploaded sheet. You can replace the questions with your own.

In [25]:
# Pick real item names from the first sheet so the demo matches your data
first_sheet_docs = next(info["docs"] for info in COLLECTIONS.values() if info["kind"] == "sheet")
item_one = first_sheet_docs[0].metadata["title"]
item_two = first_sheet_docs[min(5, len(first_sheet_docs) - 1)].metadata["title"]

demo_questions = [
    "Good evening. Can you tell me about " + item_one + "?",
    "How much is it?",
    "Does it contain any allergens?",
    "Is there anything similar you would suggest?",
    "What are your opening hours?",
    "Remind me, which dish were we talking about at the beginning?",
]

buffer_memory = new_memory()
for question in demo_questions:
    chat(question, buffer_memory, mode="buffer")
    time.sleep(2)      # small pause to respect free-tier rate limits

print("Turns stored in memory:", len(buffer_memory["history"]))

Guest: Good evening. Can you tell me about French Fries?
Assistant:
    French Fries are a classic side of crispy potato sticks, 365 kcal per serving and vegetarian. They are available in two listings:  

    - French Fries – priced at 45 EGP (Picks for you).  
    - French Fries – priced at 45 EGP (Side items).  

    Both are currently available.

Guest: How much is it?
Assistant:
    The French Fries are priced at 45 EGP each (both BR001 and BR092).

Guest: Does it contain any allergens?
Assistant:
    The listed ingredients do not identify any allergens for French Fries, but Bremer’s allergen guide notes that this is not a guarantee; guests with allergies should call 19113 to confirm.

    Allergy note: ingredient information is not an official allergen declaration. Please confirm any allergy or intolerance with the restaurant before ordering.

Guest: Is there anything similar you would suggest?
Assistant:
    If you’re looking for a similar side, you might try one of these vegetar

### Full Conversation History

In [26]:
for number, turn in enumerate(buffer_memory["history"], start=1):
    print(f"Turn {number}")
    print("  Guest    :", turn["user"])
    print("  Assistant:", turn["assistant"][:200])
    print()

Turn 1
  Guest    : Good evening. Can you tell me about French Fries?
  Assistant: French Fries are a classic side of crispy potato sticks, 365 kcal per serving and vegetarian. They are available in two listings:  

- French Fries [BR001] – priced at 45 EGP (Picks for you).  
- Fren

Turn 2
  Guest    : How much is it?
  Assistant: The French Fries are priced at 45 EGP each (both BR001 and BR092).

Turn 3
  Guest    : Does it contain any allergens?
  Assistant: The listed ingredients do not identify any allergens for French Fries, but Bremer’s allergen guide notes that this is not a guarantee; guests with allergies should call 19113 to confirm.

Turn 4
  Guest    : Is there anything similar you would suggest?
  Assistant: If you’re looking for a similar side, you might try one of these vegetarian options:

- Sautéed Vegetables [BR085] – 55 EGP  
- Noodles [BR086] – 75 EGP  
- Yellow Rice [BR087] – 50 EGP  
- Brown Rice

Turn 5
  Guest    : What are your opening hours?
  Assistant: Bran

## 9. Demo: Summary Buffer Memory (Long Conversations)

Only the last two turns are kept word for word. Everything older is compressed into a summary. The last question asks about something said at the very beginning, so the assistant can only answer from the summary.

In [27]:
long_questions = [
    "Hello, I am planning a birthday dinner for four guests.",
    "Which dishes would you recommend for a celebration?",
    "How much is " + item_two + "?",
    "Do you have any special offers or private dining options?",
    "What do you need from me to make a reservation?",
    "What was the occasion I mentioned at the start, and for how many people?",
]

summary_memory = new_memory(keep_recent=2)
for question in long_questions:
    chat(question, summary_memory, mode="summary_buffer")
    time.sleep(2)

Guest: Hello, I am planning a birthday dinner for four guests.
Assistant:
    Hello! I’d be delighted to help you plan a birthday dinner for four guests. How can I assist you with the menu or any special arrangements?

Guest: Which dishes would you recommend for a celebration?
Assistant:
    For a birthday celebration you might enjoy a mix of crowd‑pleasers and something a bit special.  
    - **Grilled Kofta Rice Dish** – hearty and flavorful.  
    - **Tandoori Chicken Rice Dish** – a classic with a spicy kick.  
    - **Cheese Lovers Pizza** – great for sharing and loved by many.  
    - **Chicken BBQ Pizza** – smoky and sweet, a good alternative.  
    - **Mushroom Steak** – a nice beef option if you want something more upscale.  

    All of these are available and should make a festive spread for your guests. If you’d like a side or drink suggestions, just let me know!

Guest: How much is Shish Tawook?
Assistant:
    Shish Tawook is priced at 95 EGP per serving. You can order it 

### Memory Status

In [28]:
print("Running summary kept by the memory:")
print(textwrap.fill(summary_memory["summary"], width=110))
print()
print("Recent turns kept in full:", len(summary_memory["history"]))

Running summary kept by the memory:
Guest is hosting a birthday dinner for four. No budget or dietary restrictions yet. The assistant recommends
five dishes: Grilled Kofta Rice Dish (BR028), Tandoori Chicken Rice Dish (BR024), Cheese Lovers Pizza (BR033),
Chicken BBQ Pizza (BR038), and Mushroom Steak (BR068). The guest asks about Shish Tawook; the assistant notes
it costs 95 EGP per serving, available as the sandwich version (BR016) or the “Picks for you” version (BR006).
No wines, prices, or decisions discussed beyond the Shish Tawook price. The assistant offers side or drink
suggestions.  New information: The assistant lists current demo offers: Family Meal Deal (10 % off for three+
Oriental Meals on Friday/Saturday dine‑in), Dessert on Us (free Nutella Cookies for dine‑in orders over
800 EGP daily), Late Night Pizza (15 % off pizzas after midnight at Branch 1), and Student Discount (10 % off
for university students at the Faculty of Law branch on Saturday‑Thursday 2‑6 PM dine‑in). N

## 10. Testing and Evaluation

The test questions are generated from your own files, so the test always matches your data:

- **Document facts** and **sheet facts:** questions about random PDF chunks and sheet rows.
- **Whole-list questions:** highest and lowest value of a numeric column. The correct answer is computed with pandas, so it is exact.
- **Memory (follow-ups):** two-question conversations where the second question uses "it" without the name.
- **Out of scope:** unrelated questions that must be politely declined.

Two metrics are measured for every answer:

- **Accuracy (0 or 1):** exact name check for pandas questions, otherwise an LLM judge compares the answer with a reference answer.
- **Relevance (1 to 5):** the LLM judge rates how directly and completely the answer addresses the question.

### Test Question Generator

In [29]:
generator_llm = make_llm(ANSWER_MODEL, temperature=0.3, max_tokens=400)
rng = random.Random(42)
QUESTIONS_PER_COLLECTION = 2
FOLLOWUP_SCENARIOS = 2

QA_PROMPT = ChatPromptTemplate.from_template(
    """You create test questions for a restaurant assistant.

Source text:
{source}

Write ONE question that a restaurant guest could ask and that is fully answered by the source text, plus its short correct answer.
The correct answer must contain ONLY the facts the question asks for, as plain facts: no disclaimers, no remarks about demo data or estimates, no extra details the question did not ask for.
The question must be self-contained and must name the dish, drink, offer or topic it is about.
Return only a JSON object with the keys "question" and "reference"."""
)

FOLLOWUP_PROMPT = ChatPromptTemplate.from_template(
    """You create test conversations for a restaurant assistant.

Source text:
{source}

Write a two-question conversation with a guest. Question 1 names the item or topic from the source text.
Question 2 is a short follow-up about the same item that uses a pronoun such as "it" or "that one" and does NOT repeat the name.
Both questions must be answerable from the source text. Give a short correct answer for each, containing only the facts that question asks for (no disclaimers, no extra details).
Return only a JSON object with the keys "q1", "a1", "q2" and "a2"."""
)

qa_chain = QA_PROMPT | generator_llm | StrOutputParser()
followup_chain = FOLLOWUP_PROMPT | generator_llm | StrOutputParser()


def read_json(text):
    return json.loads(re.search(r"\{.*\}", text, flags=re.DOTALL).group(0))


def sample_docs(docs, n, min_chars=0):
    pool = [d for d in docs if len(d.page_content) >= min_chars] or list(docs)
    return rng.sample(pool, min(n, len(pool)))

### Whole-List Questions (Answers Computed With Pandas)

In [30]:
def whole_list_scenarios():
    # Highest and lowest value of the first useful numeric column. Pandas gives the exact correct answer.
    for (file_name, tab), frame in sheet_tables.items():
        title_column = pick_title_column(frame)
        for column in frame.columns:
            is_id = column.lower() == "id" or column.lower().endswith("_id")
            if column == title_column or is_id or not pd.api.types.is_numeric_dtype(frame[column]) or frame[column].nunique() < 4:
                continue
            scenarios = []
            for word, value in (("highest", frame[column].max()), ("lowest", frame[column].min())):
                winners = frame[frame[column] == value]
                if len(winners) == 1:
                    name = str(winners.iloc[0][title_column])
                    scenarios.append({
                        "category": "Whole-list questions",
                        "turns": [{"q": f"In the {tab} list, which entry has the {word} {column.replace('_', ' ')}?",
                                   "reference": f"{name} ({value})", "must": [name]}],
                    })
            return scenarios
    return []

### Building the Test Set

In [31]:
def build_scenarios():
    scenarios = []

    # Facts from every collection
    for name, info in COLLECTIONS.items():
        category = "Document facts" if info["kind"] == "pdf" else f"Sheet facts ({name})"
        for doc in sample_docs(info["docs"], QUESTIONS_PER_COLLECTION, min_chars=200 if info["kind"] == "pdf" else 0):
            try:
                data = read_json(qa_chain.invoke({"source": doc.page_content}))
                scenarios.append({"category": category, "turns": [{"q": data["question"], "reference": data["reference"], "must": []}]})
            except Exception as error:
                print(f"Skipped one generated question for {name}: {error}")

    # Whole-list questions
    scenarios.extend(whole_list_scenarios())

    # Memory: two-question conversations with a follow-up
    sheet_docs = [d for info in COLLECTIONS.values() if info["kind"] == "sheet" for d in info["docs"]]
    for doc in sample_docs(sheet_docs, FOLLOWUP_SCENARIOS):
        try:
            data = read_json(followup_chain.invoke({"source": doc.page_content}))
            scenarios.append({"category": "Memory (follow-ups)", "turns": [
                {"q": data["q1"], "reference": data["a1"], "must": []},
                {"q": data["q2"], "reference": data["a2"], "must": []},
            ]})
        except Exception as error:
            print(f"Skipped one generated conversation: {error}")

    # Out of scope
    refusal = "The assistant politely says it can only help with questions about the restaurant."
    for question in ["Who won the football World Cup in 2018?", "Can you write a Python script for me?"]:
        scenarios.append({"category": "Out of scope", "turns": [{"q": question, "reference": refusal, "must": []}]})

    for number, scenario in enumerate(scenarios, start=1):
        scenario["id"] = f"test_{number:02d}"
    return scenarios


EVAL_SCENARIOS = build_scenarios()
print(f"Scenarios: {len(EVAL_SCENARIOS)} | Questions: {sum(len(s['turns']) for s in EVAL_SCENARIOS)}")
for scenario in EVAL_SCENARIOS:
    print(f"  {scenario['id']} [{scenario['category']}] " + " -> ".join(t["q"] for t in scenario["turns"]))

Scenarios: 13 | Questions: 15
  test_01 [Sheet facts (menu)] What is the price in Egyptian pounds and the calorie count of the BR Combo?
  test_02 [Sheet facts (menu)] What is the price of the Grilled Chicken Breasts sandwich?
  test_03 [Sheet facts (offers)] What are the details of the Student Discount offer at the Faculty of Law branch?
  test_04 [Sheet facts (offers)] What is the Late Night Pizza offer at Branch 1 (Gehan Al Sadat), and when does it apply?
  test_05 [Sheet facts (branches)] What are the opening hours of the Bremer - Faculty of Law Gate (Branch 2) restaurant?
  test_06 [Sheet facts (branches)] What are the opening hours of the Bremer - Gehan Al Sadat (Branch 1) restaurant?
  test_07 [Document facts] What is the price of the Chicken BBQ Pizza?
  test_08 [Document facts] What is the price and estimated calorie count of the Chicken Rolls?
  test_09 [Whole-list questions] In the Menu list, which entry has the lowest price egp?
  test_10 [Memory (follow-ups)] Do you have K

### Grading (Accuracy and Relevance)

In [32]:
JUDGE_PROMPT = ChatPromptTemplate.from_template(
    """You are grading the answer of a restaurant assistant.

Question: {question}
Reference answer (the correct facts): {reference}
Assistant answer: {answer}

1. "correct": true if the assistant answer gives every fact that the question asks for, in agreement with the reference, and does not contradict it. Extra details are allowed. Do NOT mark an answer wrong because it leaves out a disclaimer, a note about demo data or estimates, or a detail the question did not ask for. If the reference says the assistant should politely decline, "correct" is true when the answer declines and does not answer the question itself.
2. "relevance": rate from 1 to 5 how directly, completely and politely the answer deals with the question (5 is best, 1 is irrelevant or wrong). When the reference says the assistant should decline, a clear polite refusal that offers help with the restaurant is the perfect answer and gets 5.
Return only a JSON object with the keys "correct" (true or false), "relevance" (an integer) and "comment" (one short sentence)."""
)
judge_chain = JUDGE_PROMPT | fast_llm | StrOutputParser()


def grade(question, reference, must, answer):
    # Returns: accurate (True or False), relevance (1 to 5), judge comment
    if answer.startswith("ERROR"):
        return False, 1, "the assistant failed to answer"
    try:
        data = read_json(judge_chain.invoke({"question": question, "reference": reference, "answer": answer}))
        judged_ok = str(data.get("correct")).strip().lower() == "true"
        relevance, comment = int(data["relevance"]), str(data.get("comment", ""))
    except Exception:
        judged_ok, relevance, comment = False, None, "grading failed"

    if must:     # exact check: every required name must appear in the answer
        accurate = all(item.lower() in answer.lower() for item in must)
    else:
        accurate = judged_ok
    return accurate, relevance, comment

### Running the Evaluation

Every scenario starts with fresh memory, so follow-up questions test the memory directly. The pause between questions respects free-tier rate limits, so this takes a few minutes.

In [33]:
rows = []
for scenario in EVAL_SCENARIOS:
    memory = new_memory()
    for step, turn in enumerate(scenario["turns"], start=1):
        try:
            reply = ask(turn["q"], memory)
            answer, seconds = reply["answer"], reply["seconds"]
        except Exception as error:
            answer, seconds = f"ERROR: {error}", 0.0

        accurate, relevance, comment = grade(turn["q"], turn["reference"], turn["must"], answer)
        rows.append({
            "id": f"{scenario['id']}.{step}", "category": scenario["category"], "question": turn["q"],
            "reference": turn["reference"], "answer": answer, "accurate": int(accurate),
            "relevance": relevance, "seconds": round(seconds, 1), "judge_comment": comment,
        })
        time.sleep(8)

results = pd.DataFrame(rows)
results[["id", "category", "question", "accurate", "relevance"]]

,id,category,question,accurate,relevance
0,test_01.1,Sheet facts (menu),What is the price in Egyptian pounds and the calorie count of the BR Combo?,1,5
1,test_02.1,Sheet facts (menu),What is the price of the Grilled Chicken Breasts sandwich?,1,5
2,test_03.1,Sheet facts (offers),What are the details of the Student Discount offer at the Faculty of Law branch?,1,5
3,test_04.1,Sheet facts (offers),"What is the Late Night Pizza offer at Branch 1 (Gehan Al Sadat), and when does it apply?",1,5
4,test_05.1,Sheet facts (branches),What are the opening hours of the Bremer - Faculty of Law Gate (Branch 2) restaurant?,1,5
5,test_06.1,Sheet facts (branches),What are the opening hours of the Bremer - Gehan Al Sadat (Branch 1) restaurant?,1,5
6,test_07.1,Document facts,What is the price of the Chicken BBQ Pizza?,1,5
7,test_08.1,Document facts,What is the price and estimated calorie count of the Chicken Rolls?,1,5
8,test_09.1,Whole-list questions,"In the Menu list, which entry has the lowest price egp?",1,5
9,test_10.1,Memory (follow-ups),Do you have Kashmiri Rice on the menu?,1,5


### Results

In [34]:
print(f"Questions evaluated : {len(results)}")
print(f"Overall accuracy    : {results['accurate'].mean() * 100:.1f} percent")
print(f"Average relevance   : {results['relevance'].mean():.2f} out of 5")
print(f"Average latency     : {results['seconds'].mean():.1f} seconds per answer")

by_category = results.groupby("category").agg(
    questions=("id", "count"),
    accuracy_percent=("accurate", lambda s: round(s.mean() * 100, 1)),
    avg_relevance=("relevance", "mean"),
).round(2)
by_category

Questions evaluated : 15
Overall accuracy    : 100.0 percent
Average relevance   : 5.00 out of 5
Average latency     : 8.3 seconds per answer


,questions,accuracy_percent,avg_relevance
category,,,
Document facts,2,100.0,5.0
Memory (follow-ups),4,100.0,5.0
Out of scope,2,100.0,5.0
Sheet facts (branches),2,100.0,5.0
Sheet facts (menu),2,100.0,5.0
Sheet facts (offers),2,100.0,5.0
Whole-list questions,1,100.0,5.0


### Failed Cases

In [35]:
failed = results[results["accurate"] == 0]
if failed.empty:
    print("Every answer matched its reference.")
for _, row in failed.iterrows():
    print(f"{row['id']} | {row['question']}")
    print(f"  reference: {row['reference']}")
    print(textwrap.indent(textwrap.fill(row["answer"], width=100), "  answer: "))
    print(f"  judge: {row['judge_comment']}\n")

Every answer matched its reference.


### Photo Check

Run this once before the demo. It tests every photo link and lists photos that are shared by different dishes. Then it shows every photo with its id and name, so you can confirm by eye that each photo matches its dish. If one is wrong, put the right link in `IMAGE_OVERRIDES` (Section 4) as `{"BR028": "https://..."}` and run the cells again.

In [36]:
from IPython.display import HTML, display
from concurrent.futures import ThreadPoolExecutor


def check_link(item):
    try:
        response = requests.get(item["url"], timeout=20, stream=True, headers={"User-Agent": "Mozilla/5.0"})
        ok = response.status_code == 200 and response.headers.get("content-type", "").startswith("image")
        response.close()
        return item["id"], ok
    except Exception:
        return item["id"], False


with ThreadPoolExecutor(max_workers=8) as pool:
    link_status = dict(pool.map(check_link, [m for m in MENU_ITEMS.values() if m["url"]]))
broken = [i for i, ok in link_status.items() if not ok]
print(f"Photo links tested: {len(link_status)} | not working: {broken or 'none'}")

# The same photo used by dishes with different names is a likely mistake
by_url = {}
for item in (m for m in MENU_ITEMS.values() if m["url"]):
    by_url.setdefault(item["url"], []).append(item)
for url, items in by_url.items():
    if len({i["name"] for i in items}) > 1:
        print("Same photo, different dishes:", [f"{i['id']} {i['name']}" for i in items])

# Gallery for a visual check
cards = "".join(
    f'<figure style="margin:0;width:150px;text-align:center;font:12px sans-serif">'
    f'<img src="{i["url"]}" referrerpolicy="no-referrer" style="width:150px;height:110px;object-fit:cover;border-radius:10px;'
    f'{"outline:3px solid red;" if not link_status.get(i["id"], False) else ""}">'
    f'<figcaption><b>{i["id"]}</b> {i["name"]}<br><small>{i["category"]}</small></figcaption></figure>'
    for i in MENU_ITEMS.values()
)
display(HTML(f'<div style="display:flex;flex-wrap:wrap;gap:12px">{cards}</div>'))

Photo links tested: 111 | not working: none


### Finding a Photo for Items Without One

Items whose `image_url` is empty in the sheet get a photo from the web. It only works on those few items, so it takes under a minute:

1. Search the web for the dish name and keep up to 20 candidate pictures.
2. Drop pictures that are small, black or flat, or that CLIP says are not food.
3. CLIP compares each remaining picture with the names of the dishes in the same category. The picture that looks most like **this** dish wins, and only if it is clearly the right dish.

The chosen pictures are shown so you can confirm them. Then copy the printed link into the sheet's `image_url` column. If nothing trustworthy is found, the item simply has no picture.

In [37]:
import unicodedata
import numpy as np
from io import BytesIO
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from sentence_transformers import SentenceTransformer, util
from IPython.display import HTML, display

try:
    from ddgs import DDGS
except ImportError:
    from duckduckgo_search import DDGS

HEADERS = {"User-Agent": "Mozilla/5.0"}
NO_PHOTO = [i for i, m in MENU_ITEMS.items() if not m["url"]]
print("Items without a photo:", [f"{i} {MENU_ITEMS[i]['name']}" for i in NO_PHOTO])

if NO_PHOTO:
    clip = SentenceTransformer("sentence-transformers/clip-ViT-B-32")
    NOT_FOOD = ["a black image", "an empty plate", "a logo or a text graphic", "a photo of a person", "a photo of a room or a building", "a screenshot"]
    FOOD = ["a photo of a plated restaurant dish", "a photo of food"]
    label_vectors = clip.encode(FOOD + NOT_FOOD, convert_to_tensor=True, normalize_embeddings=True)

    def download(url):
        try:
            response = requests.get(url, timeout=10, headers=HEADERS)
            if response.status_code == 200 and len(response.content) < 6_000_000:
                return Image.open(BytesIO(response.content)).convert("RGB")
        except Exception:
            pass
        return None

    def usable(image):
        if image is None or min(image.size) < 300:
            return False
        pixels = np.asarray(image.resize((96, 96))).astype(float)
        return pixels.mean() > 40 and (pixels.max(axis=2) < 30).mean() < 0.8 and pixels.std() > 18

    def find_photo(item_id):
        item = MENU_ITEMS[item_id]
        queries = [f"{item['name']} {item['category']} dish", f"Bremer Mansoura {item['name']}", f"{item['name']} egyptian restaurant"]
        links = []
        for query in queries:
            try:
                links += [hit.get("image") for hit in DDGS().images(query, max_results=10) if hit.get("image", "").startswith("http")]
            except Exception:
                pass
        links = list(dict.fromkeys(links))[:20]
        with ThreadPoolExecutor(max_workers=8) as pool:
            images = list(pool.map(download, links))
        good = [(link, image) for link, image in zip(links, images) if usable(image)]
        if not good:
            return None, 0.0

        # the picture must look like this dish rather than the other dishes of the same category
        rivals = sorted({m["name"] for m in MENU_ITEMS.values() if m["category"] == item["category"]} | {item["name"]})
        rival_vectors = clip.encode([f"a photo of {n}, a restaurant dish" for n in rivals], convert_to_tensor=True, normalize_embeddings=True)
        vectors = clip.encode([image for _, image in good], convert_to_tensor=True, normalize_embeddings=True, batch_size=16)
        name_prob = (100 * util.cos_sim(vectors, rival_vectors)).softmax(dim=1)[:, rivals.index(item["name"])]
        food_prob = (100 * util.cos_sim(vectors, label_vectors)).softmax(dim=1)[:, :len(FOOD)].sum(dim=1)
        score = name_prob * (food_prob > 0.5)
        best = int(score.argmax())
        return (good[best][0], float(score[best])) if float(score[best]) >= 0.35 else (None, float(score[best]))

    FOUND = {}
    for item_id in NO_PHOTO:
        link, score = find_photo(item_id)
        print(f"{item_id} {MENU_ITEMS[item_id]['name']}: {'found (match %.2f)' % score if link else 'nothing trustworthy found'}")
        if link:
            FOUND[item_id] = link
            MENU_ITEMS[item_id]["url"] = link

    if FOUND:
        display(HTML('<div style="display:flex;gap:14px;flex-wrap:wrap">' + "".join(
            f'<figure style="margin:0;width:220px;font:12px sans-serif;text-align:center"><img src="{u}" referrerpolicy="no-referrer" '
            f'style="width:220px;height:165px;object-fit:cover;border-radius:10px"><figcaption><b>{i}</b> {MENU_ITEMS[i]["name"]}</figcaption></figure>'
            for i, u in FOUND.items()) + "</div>"))
        print("\nLinks to put in the image_url column of the sheet (or in IMAGE_OVERRIDES):")
        print(json.dumps(FOUND, indent=1))

Items without a photo: []


### Photo Test: Does Every Mentioned Dish Get Its Picture?

Generic questions built from the sheet's own categories (no dish names are hard coded). For every answer:

- **Coverage:** every menu item named in the answer text must have its picture.
- **Precision:** every picture shown must belong to an item named in the answer.

In [38]:
photo_questions = []
categories = sorted({i["category"] for i in MENU_ITEMS.values() if i["category"]})
for category in random.Random(7).sample(categories, min(4, len(categories))):
    photo_questions.append(f"What do you have in {category}? Give me prices.")
photo_questions += [
    "I want a rice dish, a salad and something to drink. Choose for me.",
    "Suggest something for a family of four with different tastes.",
    "What is the cheapest main dish?",
]

covered = shown_right = named = shown = 0
for question in photo_questions:
    reply = ask(question, new_memory())
    clean = reply["answer"]
    offered = {i for s, e, ids in find_mentions(clean) if is_presented(clean, s, e) for i in ids}
    offered_urls = {MENU_ITEMS[i]["url"] for i in offered if MENU_ITEMS[i]["url"]}
    shown_urls = {MENU_ITEMS[i]["url"] for i in reply["item_ids"]}
    named += len(offered_urls); covered += len(offered_urls & shown_urls)
    shown += len(shown_urls); shown_right += len(shown_urls & offered_urls)
    print(f"{question[:55]:<57} offered {len(offered_urls):>2} | pictures {len(shown_urls):>2}")
    time.sleep(6)
print(f"\nCoverage (offered items with a picture): {covered}/{named} | Precision (pictures of offered items): {shown_right}/{shown}")

What do you have in Rice Dishes? Give me prices.          offered  5 | pictures  5
What do you have in Beef? Give me prices.                 offered  2 | pictures  2
What do you have in Oriental Meals? Give me prices.       offered  7 | pictures  7
What do you have in Soft Drinks? Give me prices.          offered  4 | pictures  4
I want a rice dish, a salad and something to drink. Cho   offered  3 | pictures  3
Suggest something for a family of four with different t   offered  4 | pictures  4
What is the cheapest main dish?                           offered  1 | pictures  1

Coverage (offered items with a picture): 26/26 | Precision (pictures of offered items): 26/26


## 11. Simple Chat in the Notebook

Chat with the assistant directly in a notebook cell, with no interface: type a question, press Enter, read the answer, and repeat. The assistant remembers the whole chat (the memory type is set by `MEMORY_MODE` in the Settings cell). Type `exit` to stop.

In [39]:
chat_memory = new_memory()
print("Type your question and press Enter. Type 'exit' to stop.\n")

while True:
    question = input("You: ").strip()
    if question.lower() in ("exit", "quit"):
        print("Goodbye!")
        break
    if not question:
        continue
    try:
        reply = ask(question, chat_memory, MEMORY_MODE)
        print("Assistant:", textwrap.indent(reply["answer"], "    ").lstrip(), "\n")
    except Exception as error:
        print("Sorry, something went wrong while preparing the answer:", error, "\n")

Type your question and press Enter. Type 'exit' to stop.

You: exit
Goodbye!


## 12. Chat Interface

A Gradio chat window turns the notebook into a working demo, with an orange theme, a welcome message, Arabic and English examples, and a photo card for every dish that the assistant mentions in its answer (the photo links come from the `image_url` column of the Menu sheet). Each browser session gets its own memory, and clearing the chat resets it. The memory type is set by `MEMORY_MODE` in the Settings cell. Run the cell and open the link that is printed.

In [40]:
import gradio as gr

SESSIONS = {}      # one memory per browser session


def photo_cards(item_ids):
    # One card per menu item the assistant presented, taken by item_id (never by guessing from the text)
    cards = ""
    for item_id in item_ids:
        item = MENU_ITEMS[item_id]
        if not item["url"]:
            continue      # no trustworthy photo for this item: show nothing rather than a wrong picture
        price = f" · {item['price']} EGP" if item["price"] else ""
        cards += (
            f'<figure style="margin:0;text-align:center;width:150px">'
            f'<img src="{item["url"]}" alt="{item["name"]}" referrerpolicy="no-referrer" '
            f'style="width:150px;height:110px;object-fit:cover;border-radius:12px">'
            f'<figcaption style="font-size:12px">{item["name"]}{price}</figcaption></figure>'
        )
    return f'\n\n<div style="display:flex;gap:10px;flex-wrap:wrap">{cards}</div>' if cards else ""


def respond(message, history, request: gr.Request = None):
    key = getattr(request, "session_hash", None) or "default"
    if not history or key not in SESSIONS:
        SESSIONS[key] = new_memory()      # a new or cleared chat starts with empty memory
    try:
        reply = ask(message, SESSIONS[key], MEMORY_MODE)
    except Exception as error:
        return f"Sorry, something went wrong while preparing the answer: {error}"
    return reply["answer"] + photo_cards(reply["item_ids"])      # pictures are added to the displayed answer only


demo = gr.ChatInterface(
    fn=respond,
    title=f"{RESTAURANT_NAME} Virtual Concierge",
    examples=["What vegetarian dishes do you serve?", "What are your opening hours?", "Do you have any special offers?"],
    cache_examples=False,
)
demo.launch(share=True)      # share=True prints a public link


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://46324c690a6094dc94.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 13. Summary

**What was built**

- A knowledge base from PDF files (unstructured) and Excel / CSV files or a Google Sheet (structured). Every sheet tab is its own collection.
- HuggingFace multilingual embeddings stored in FAISS, one saved index per collection.
- A RAG pipeline with a query planner (rewrites follow-ups, routes to the right collections, handles whole-list questions), grounded answers, an allergy note, and polite refusal of unrelated questions.
- Buffer memory (full history) and summary buffer memory (summary plus recent turns).
- An evaluation of accuracy and relevance on a test set generated from the same files.
- A Gradio chat interface where every browser session has its own memory.

**Limitations**

- Free-tier Groq rate limits slow the evaluation; the answer model falls back to the small model when its limit is hit.
- Whole-list questions put an entire sheet in the prompt, fine for a menu but not for thousands of rows.
- Scanned PDFs (images without text) need OCR first.
- Memory lives in the notebook session and is lost when the runtime restarts.

**Future work**

- Compute numeric questions (cheapest, average price) with a pandas or SQL tool.
- Hybrid search (keyword plus vector) with a reranker.
- A reservation tool, and memory stored in a database across visits.